# Chương 7: Hệ đa agent: phối hợp, phản biện và so với một RAG đơn

        **Câu hỏi của chương.** Khi nào nhiều agent đáng giá hơn một RAG? Câu trả lời của chương này: chỉ khi (1) câu hỏi cần nhiều loại năng lực khác nhau
        (số liệu đo từ dữ liệu, tính toán chính xác, giải thích từ tài liệu) và (2) có một bước kiểm tra độc lập bằng bằng chứng chạy được, chứ không phải
        một mô hình tự đồng ý với chính nó. Với câu hỏi một bước như "Insulin do cơ quan nào tạo ra?", một RAG đơn là đủ; notebook sẽ chạy cả hai loại câu hỏi để thấy rõ điều này.

        **Các agent.** Điều phối (lập kế hoạch bằng luật), thống kê (pandas trên CDC BRFSS 2023, 150.000 người), tính toán (`safe_calculate`), truy hồi (kho NIDDK/CDC/MedlinePlus),
        đồ thị (tra nguồn trong đồ thị tri thức, dùng cho tài chính), viết (mô hình Qwen2.5-1.5B-Instruct viết nháp tự do) và **phản biện**: một bộ kiểm tra chạy được, không phải mô hình ngôn ngữ,
        trích mọi con số trong bản nháp rồi đối chiếu từng số với kết quả công cụ và đoạn tài liệu. Có lỗi thì gửi Message yêu cầu agent viết sửa; tối đa 2 vòng.

        Demo localhost ở `/agents` (cổng 8044) chạy cùng mã này. Các ô bên dưới có nhãn AGENT-01 đến AGENT-15.

In [ ]:
# Cell AGENT-01
import os, sys, json, time, inspect, copy
from pathlib import Path
for key in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[key] = '2'
os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'src/tieuluan').exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
OUT = ROOT/'results/tieuluan/complete/notebooks'
OUT.mkdir(parents=True, exist_ok=True)
print('Project:', ROOT)

if os.environ.get('CH5_LOWMEM') == '1':
    import scripts.chapter05.lowmem_llm as lowmem; lowmem.install()
    print('Nạp Qwen ở chế độ tiết kiệm bộ nhớ (int8), xem lowmem_llm.py')

## 1. State, Message và các hàm kiểm số

        `State` là bảng trạng thái chung: câu hỏi, kế hoạch, sự kiện của công cụ (`facts`, `derived`), các đoạn tài liệu, các bản nháp (`drafts`), các lần phản biện (`reviews`) và danh sách `messages`.
        Mỗi agent không gọi trực tiếp agent khác mà ghi một `Message(sender, receiver, kind, payload)` vào State (kế hoạch, sự kiện, bản nháp, yêu cầu sửa, kết luận).
        `extract_numbers` tìm mọi con số (nhận cả định dạng Việt "14,7%" và "150.000", lẫn Anh "37.3 million"); `close` cho phép làm tròn hợp lý (số viết với d chữ số thập phân thì sai lệch tối đa nửa đơn vị cuối).

In [ ]:
# Cell AGENT-02
"""Hệ đa agent Chương 7: điều phối, thống kê, truy hồi, tính toán, đồ thị, viết và phản biện.

Nguyên tắc thiết kế: đa agent chỉ đáng giá khi câu hỏi cần nhiều loại năng lực (số liệu từ dữ liệu, giải thích từ tài liệu,
tính toán chính xác) và có kiểm tra độc lập bằng bằng chứng chạy được. Câu hỏi một bước thì một RAG đơn là đủ.

Các agent giao tiếp bằng Message ghi trong State. Agent phản biện không phải mô hình ngôn ngữ: nó trích mọi con số trong bản nháp
và đối chiếu với sự kiện của công cụ và với các đoạn tài liệu. Có lỗi thì gửi yêu cầu sửa cho agent viết; tối đa 2 vòng sửa.
"""
import re
import time
from dataclasses import dataclass, field

MAX_REVISIONS = 2
SAMPLE_SIZE_NOTE_VI = ('Đây là số liệu của mẫu khảo sát CDC BRFSS 2023 gồm {n} người, chưa hiệu chỉnh trọng số, '
                       'nên chỉ mô tả mẫu này chứ không đại diện cho toàn dân Mỹ.')
SAMPLE_SIZE_NOTE_EN = ('These figures describe the CDC BRFSS 2023 survey sample of {n} people, without survey weights, '
                       'so they describe this sample and do not represent the whole US population.')


@dataclass
class Message:
    sender: str
    receiver: str
    kind: str
    payload: dict


@dataclass
class State:
    question: str
    domain: str
    report: str | None = None
    plan: dict = field(default_factory=dict)
    facts: list = field(default_factory=list)        # sự kiện từ agent thống kê
    derived: list = field(default_factory=list)      # kết quả của agent tính toán
    evidence: list = field(default_factory=list)     # các đoạn tài liệu truy hồi
    explanation: list = field(default_factory=list)  # câu giải thích rút ra, có mã nguồn
    graph_context: list = field(default_factory=list)
    drafts: list = field(default_factory=list)       # [{'round', 'by', 'text'}]
    reviews: list = field(default_factory=list)      # kết quả mỗi lần phản biện
    answer: str = ''
    english_answer: str = ''
    status: str = 'pending'
    messages: list = field(default_factory=list)
    query: str = ''
    language: str = ''
    steps: list = field(default_factory=list)
    dialogue: list = field(default_factory=list)

    def send(self, sender, receiver, kind, **payload):
        self.messages.append(Message(sender, receiver, kind, payload))


# ---------------------------------------------------------------------------------------------
# Tiện ích số học và văn bản
# ---------------------------------------------------------------------------------------------
def norm(text):
    from src.chapter04.stats import strip_accents
    return ' ' + strip_accents(text.lower()).rstrip('?. ') + ' '


NUM = re.compile(r'(?<![A-Za-z0-9])(\d{1,3}(?:[.,]\d{3})+(?![\d])|\d+(?:[.,]\d+)?)(\s*%|\s*(?:million|billion|thousand|triệu|tỷ|tỉ|nghìn|ngàn|lần|times|điểm phần trăm|percentage points?)\b)?',
                 re.I)
MULT = {'million': 1e6, 'triệu': 1e6, 'billion': 1e9, 'tỷ': 1e9, 'tỉ': 1e9, 'thousand': 1e3, 'nghìn': 1e3, 'ngàn': 1e3}
DOC_MARK = re.compile(r'according|niddk|cdc\b|medlineplus|the (?:passages?|documents?|websites?)|theo |tài liệu|nguồn', re.I)
SAMPLE_MARK = re.compile(r'\bsample\b|survey|brfss|respondents?|150[.,]?000|mẫu|khảo sát|dữ liệu|people in the data', re.I)
LIMIT_MARK = re.compile(r'sample|mẫu|brfss|survey|khảo sát|unweighted|không có trọng số|chưa hiệu chỉnh', re.I)


def parse_number(text, unit):
    unit = (unit or '').strip().lower()
    if re.fullmatch(r'\d{1,3}(?:[.,]\d{3})+', text):
        return float(re.sub(r'[.,]', '', text)), 0
    clean = text.replace(',', '.')
    decimals = len(clean.split('.')[1]) if '.' in clean else 0
    return float(clean) * MULT.get(unit, 1), decimals


def sentences_of(text):
    parts = re.split(r'(?<=[.!?])\s+|\n+', text.strip())
    return [p for p in parts if p.strip()]


def extract_numbers(text, ignore=()):
    """Trả về danh sách {'text','value','decimals','percent','scale','sentence'} cho các số cần có bằng chứng."""
    out = []
    skip_text = ' '.join(str(x) for x in ignore)
    skip_numbers = {m.group(1) for m in NUM.finditer(skip_text)}
    for sentence in sentences_of(text):
        for m in NUM.finditer(sentence):
            raw, unit = m.group(1), (m.group(2) or '').strip()
            before = sentence[max(0, m.start() - 6):m.start()].lower()
            if re.search(r'(type|típ|tip|loại)\s*$', before) or re.match(r'^\s*[-*]?\s*$', sentence[:m.start()]) and re.match(r'^\d\.?$', raw) and not unit:
                continue
            value, decimals = parse_number(raw, unit.lower())
            percent = unit.startswith('%')
            if not unit and decimals == 0 and (1900 <= value <= 2100 or 0 <= value <= 3):
                continue  # năm, số đếm nhỏ
            if raw in skip_numbers:
                continue
            scale = MULT.get(unit.lower(), 1)
            out.append({'text': (raw + (' ' + unit if unit else '')).strip(), 'value': value, 'decimals': decimals,
                        'percent': percent, 'scale': scale, 'sentence': sentence})
    return out


def close(x, decimals, scale, v):
    """x (viết với `decimals` chữ số thập phân, nhân `scale`) khớp v khi làm tròn hợp lý."""
    tol = 0.5 * (10 ** -decimals) * scale + 1e-9
    return abs(x - v) <= tol or (v != 0 and abs(x - v) / abs(v) <= 0.002)


def fact_numbers(state):
    """Mọi số mà công cụ đã cung cấp, kèm nhãn nguồn."""
    out = []
    for f in state.facts + state.derived:
        out.append((f['value'], f['id'], f))
        for key in ('numerator', 'denominator'):
            if f.get(key) is not None:
                out.append((float(f[key]), f['id'], f))
    return out

## 2. Agent điều phối (Planner)

        Dùng luật và từ khóa (đã bỏ dấu) để quyết định: câu hỏi ngoài phạm vi thì từ chối ngay; câu chỉ cần kiến thức thì đánh dấu "đơn giản: một RAG là đủ" và chỉ gọi truy hồi, viết, phản biện;
        câu cần số liệu thì lập danh sách sự kiện cần tính (một tỉ lệ, hai nhóm, nhóm cao nhất và thấp nhất...) cùng một phép tính (chia hoặc trừ). Kế hoạch này xác định, nhanh, không dùng mô hình sinh.

In [ ]:
# Cell AGENT-03
# ---------------------------------------------------------------------------------------------
# Agent điều phối (Planner)
# ---------------------------------------------------------------------------------------------
HEALTH_WORDS = ['tieu duong', 'diabet', 'insulin', 'duong huyet', 'duong trong mau', 'glucose', 'blood sugar', 'a1c', 'huyet ap', 'cholesterol', 'beo phi', 'bmi',
                'tuy', 'pancrea', 'suc khoe', 'benh', 'thuoc', 'tim', 'than ', 'mat ', 'than kinh', 'can nang', 'an uong', 'van dong', 'hut thuoc', 'dot quy',
                'metformin', 'bien chung', 'trieu chung', 'xet nghiem', 'thai ky', 'bnr', 'brfss', 'khao sat', 'gia dinh', 'tuoi', 'nam gioi', 'nu gioi',
                'hypoglyc', 'hyperglyc', 'cgm', 'chan', 'nguoi mac', 'chan doan', 'nguy co', 'mac benh', 'health', 'disease', 'symptom', 'diagnos']
FINANCE_WORDS = ['doanh thu', 'loi nhuan', 'cong ty', 'bao cao', 'revenue', 'income', 'visa', 'mastercard', 'apple', 'microsoft', 'tai chinh', 'co phieu', 'amazon', 'google',
                 'nvidia', 'tesla', 'meta', 'jpmorgan', '10-k', 'lĩnh vực', 'linh vuc', 'hoat dong']
EXPLAIN_WORDS = ['vi sao', 'tai sao', 'giai thich', 'nguyen nhan', 'la gi', 'the nao', 'co nghia', 'nen lam', 'trieu chung', 'bien chung', 'co the gay', 'tac dong',
                 'why', 'explain', 'what is', 'how does', 'cach nao', 'lam sao', 'co anh huong', 'dieu gi', 'yeu to nguy co', 'co lien quan']
RATIO_WORDS = ['gap may lan', 'gap bao nhieu lan', 'bao nhieu lan', 'gap doi', 'gap nhieu lan', 'lan so voi', 'gap ', 'times as']
GAP_WORDS = ['chenh lech', 'khac nhau bao nhieu', 'hon bao nhieu', 'cao hon bao nhieu', 'thap hon bao nhieu', 'khoang cach', 'gap between']


class PlannerAgent:
    """Đọc câu hỏi, phân loại và lập kế hoạch bằng luật và từ khóa (xác định, không dùng mô hình sinh)."""
    role = 'Agent điều phối'
    description = 'Đọc câu hỏi, quyết định cần những agent nào và gọi theo thứ tự.'

    def __init__(self, stats):
        self.stats = stats

    def act(self, state):
        t0 = time.perf_counter()
        q = norm(state.question)
        plan = {'scope': 'diabetes', 'mode': 'simple', 'facts': [], 'calc': None, 'explain': True, 'explain_question': state.question, 'agents': []}
        if state.domain == 'finance':
            plan.update(scope='finance', summary='Câu hỏi về công ty/báo cáo tài chính: cần truy hồi đoạn báo cáo và tra đồ thị để biết đoạn thuộc công ty nào.',
                        agents=['RetrievalAgent', 'GraphAgent', 'WriterAgent', 'CriticAgent'])
        elif not any(w in q for w in HEALTH_WORDS) and not any(w in q for w in FINANCE_WORDS):
            plan.update(scope='off_topic', mode='off_topic', explain=False, agents=[],
                        summary='Câu hỏi không liên quan đến tiểu đường hay sức khỏe nên kho tài liệu và dữ liệu khảo sát không thể trả lời. Nhóm từ chối ngay, không tốn công truy hồi.')
        else:
            self.plan_diabetes(state, q, plan)
        plan['mode_label'] = {'simple': 'Đơn giản: một RAG là đủ', 'data': 'Phức hợp: cần số liệu thật', 'off_topic': 'Ngoài phạm vi'}.get(plan['mode'], plan['mode'])
        state.plan = plan
        state.send('planner', 'coordinator', 'plan', mode=plan['mode'], agents=plan['agents'], facts=len(plan['facts']), calc=bool(plan['calc']))
        state.steps.append({'agent': 'PlannerAgent', 'role': self.role, 'role_description': self.description, 'summary': plan['summary'], 'outcome': plan['mode_label'],
                            'seconds': time.perf_counter() - t0, 'evidence': {'plan': plan}})

    def plan_diabetes(self, state, q, plan):
        info = self.stats.parse(state.question) if hasattr(self.stats, 'parse') else None
        explain_hit = next((w for w in EXPLAIN_WORDS if w in q), None)
        statistical = bool(info) and (info['metric'] in ('rate', 'count', 'mean_bmi') or info.get('compare') or info.get('group_by'))
        if statistical and not self.stats.looks_statistical(state.question):
            statistical = False
        extreme_dim = next((d for kw, d in [(' nhom tuoi', 'age_group'), (' nhom thu nhap', 'income_group'), (' hoc van', 'education'), (' nhom bmi', 'bmi_category')] if kw in q), None)
        extreme = bool(extreme_dim) and any(w in q for w in ['cao nhat', 'thap nhat', 'nhieu nhat', 'it nhat'])
        mean_bmi = 'bmi' in q and any(w in q for w in ['trung binh', 'average', 'mean'])
        two_diabetes = mean_bmi and any(w in q for w in ['khong mac', 'khong bi', 'so voi nguoi khong', 'khong tieu duong'])
        if not (statistical or extreme or two_diabetes):
            plan.update(summary='Câu hỏi hỏi về kiến thức chung, không cần số liệu từ dữ liệu khảo sát và không cần tính toán. Một RAG đơn (truy hồi rồi viết) là đủ, '
                                'nên nhóm chỉ gọi agent truy hồi và agent viết, rồi agent phản biện soát số trong câu trả lời. Đây là trường hợp nhóm agent không có lợi thế.',
                        agents=['RetrievalAgent', 'WriterAgent', 'CriticAgent'])
            return
        plan['mode'] = 'data'
        target = (info or {}).get('target') or 'diabetes'
        filters = list((info or {}).get('filters') or [])
        facts, calc = [], None
        metric = 'mean_bmi' if mean_bmi else ((info or {}).get('metric') if (info or {}).get('metric') in ('rate', 'count') else 'rate')
        if two_diabetes:
            facts = [{'kind': 'mean_bmi', 'target': 'diabetes', 'filters': [], 'extra': 'diabetes_yes', 'short_vi': 'nhóm mắc tiểu đường', 'short_en': 'the diabetes group'},
                     {'kind': 'mean_bmi', 'target': 'diabetes', 'filters': [], 'extra': 'diabetes_no', 'short_vi': 'nhóm không mắc tiểu đường', 'short_en': 'the group without diabetes'}]
            calc = {'op': 'subtract', 'a': 'F1', 'b': 'F2', 'unit': 'BMI'}
            if any(w in q for w in RATIO_WORDS[:6]):
                calc = {'op': 'divide', 'a': 'F1', 'b': 'F2', 'unit': 'BMI'}
        elif 'khong van dong' in q and any(w in q for w in ['co van dong', 'nguoi van dong', 'nguoi co tap', 'co tap the duc']):
            facts = [{'kind': 'rate', 'target': target, 'filters': [('cond', 'inactive')], 'short_vi': 'nhóm không vận động', 'short_en': 'the inactive group'},
                     {'kind': 'rate', 'target': target, 'filters': [('cond', 'active')], 'short_vi': 'nhóm có vận động', 'short_en': 'the active group'}]
            calc = {'op': 'divide' if any(w in q for w in RATIO_WORDS) else 'subtract', 'a': 'F1', 'b': 'F2', 'unit': '%'}
        elif extreme:
            dim = extreme_dim
            facts = [{'kind': 'extreme', 'which': 'top', 'dim': dim, 'target': target, 'filters': [f for f in filters if f[0] != 'age'], 'short_vi': 'nhóm cao nhất', 'short_en': 'the highest age group'},
                     {'kind': 'extreme', 'which': 'low', 'dim': dim, 'target': target, 'filters': [f for f in filters if f[0] != 'age'], 'short_vi': 'nhóm thấp nhất', 'short_en': 'the lowest age group'}]
            calc = {'op': 'divide', 'a': 'F1', 'b': 'F2', 'unit': '%'}
        elif info and info.get('compare'):
            dim, vals = info['compare']
            for v in vals[:2]:
                facts.append({'kind': metric if metric != 'mean_bmi' else 'mean_bmi', 'target': target, 'filters': filters + [('dim', dim, [v])], 'short_vi': '', 'short_en': ''})
            if len(facts) == 2:
                calc = {'op': 'divide' if any(w in q for w in RATIO_WORDS) else 'subtract', 'a': 'F1', 'b': 'F2', 'unit': 'BMI' if metric == 'mean_bmi' else '%'}
        elif info and info.get('group_by') and info['group_by'] in ('age_group', 'sex', 'bmi_category', 'smoker_status', 'income_group', 'education', 'general_health'):
            facts = [{'kind': 'extreme', 'which': 'top', 'dim': info['group_by'], 'target': target, 'filters': filters, 'short_vi': 'nhóm cao nhất', 'short_en': 'the highest group'},
                     {'kind': 'extreme', 'which': 'low', 'dim': info['group_by'], 'target': target, 'filters': filters, 'short_vi': 'nhóm thấp nhất', 'short_en': 'the lowest group'}]
            calc = {'op': 'divide', 'a': 'F1', 'b': 'F2', 'unit': '%'}
        else:
            facts = [{'kind': metric, 'target': target, 'filters': filters, 'short_vi': '', 'short_en': ''}]
            gap = any(w in q for w in GAP_WORDS + ['diem phan tram'])
            if filters and metric == 'rate' and any(w in q for w in RATIO_WORDS + GAP_WORDS + ['ca mau', 'so voi ca', 'chung']):
                facts.append({'kind': 'rate', 'target': target, 'filters': [], 'short_vi': 'cả mẫu', 'short_en': 'the whole sample'})
                ratio = any(w in q for w in RATIO_WORDS)
                calc = {'op': 'subtract' if (gap and not ratio) else 'divide', 'a': 'F1', 'b': 'F2', 'unit': '%'}
        for i, spec in enumerate(facts, 1):
            spec['id'] = f'F{i}'
        plan['facts'] = facts
        plan['calc'] = ({**calc, 'id': 'D1'} if calc else None)
        plan['explain'] = bool(explain_hit)
        if explain_hit:
            idx = q.find(explain_hit)
            rest = norm(state.question)[idx:]
            raw = state.question
            marker = re.search(r'(vì sao|tại sao|giải thích|nguyên nhân|là gì|thế nào|why|explain)', raw, re.I)
            plan['explain_question'] = raw[marker.start():].strip() if marker and len(raw) - marker.start() >= 12 else raw
        parts = ['Câu hỏi cần số liệu thật từ dữ liệu khảo sát (agent thống kê)']
        if calc:
            parts.append('một phép tính chính xác (agent tính toán, không để mô hình tự tính)')
        if plan['explain']:
            parts.append('một phần giải thích từ tài liệu (agent truy hồi)')
        plan['agents'] = ['StatsAgent'] + (['CalculatorAgent'] if calc else []) + (['RetrievalAgent'] if plan['explain'] else []) + ['WriterAgent', 'CriticAgent']
        plan['summary'] = (', '.join(parts) + '. Một RAG đơn chỉ đọc tài liệu nên không có các con số này; agent phản biện sẽ đối chiếu từng số trong bản nháp với công cụ.')

## 3. Agent thống kê

        Mỗi sự kiện được tính bằng pandas trên mẫu BRFSS (qua `DiabetesStats.apply_filters`) và trả về cấu trúc: giá trị, tử số, mẫu số, nhãn tiếng Việt và tiếng Anh, câu mô tả, nguồn.
        Số này là số đo, không phải số mô hình nhớ.

In [ ]:
# Cell AGENT-04
# ---------------------------------------------------------------------------------------------
# Agent thống kê
# ---------------------------------------------------------------------------------------------
class StatsAgent:
    role = 'Agent thống kê'
    description = 'Tính số liệu thật từ dữ liệu khảo sát CDC BRFSS 2023 (150.000 người) bằng pandas.'

    def __init__(self, stats):
        self.stats = stats

    def subset(self, spec):
        from src.chapter04.stats import CONDITIONS
        df, dvi, den = self.stats.apply_filters({'filters': spec['filters']})
        extra = spec.get('extra')
        if extra == 'diabetes_yes':
            df, dvi, den = df[df.diabetes_status == 'yes'], dvi + ['mắc tiểu đường'], den + ['with diabetes']
        elif extra == 'diabetes_no':
            df, dvi, den = df[df.diabetes_status == 'no'], dvi + ['không mắc tiểu đường'], den + ['without diabetes']
        return df, dvi, den

    def evaluate(self, spec, state):
        from src.chapter04.stats import fmt_pct, fmt_int, fmt_num, CONDITIONS, DIMENSIONS
        stats = self.stats
        spec = dict(spec)
        dim_label = None
        if spec['kind'] == 'extreme':
            df, dvi, den = self.subset(spec)
            col = DIMENSIONS[spec['dim']]['col']
            grouped = stats.target_mask(df, spec['target']).groupby(df[col])
            table = [(k, float(v) * 100, int(n), int(c)) for k, v, n, c in zip(grouped.mean().index, grouped.mean(), grouped.size(), grouped.sum())]
            table = [r for r in table if r[2] >= 200]
            pick = max(table, key=lambda r: r[1]) if spec['which'] == 'top' else min(table, key=lambda r: r[1])
            group, pct, n, k = pick
            dim_label = (spec['dim'], group)
            spec = {**spec, 'kind': 'rate', 'filters': list(spec['filters']) + [('dim', spec['dim'], [group])] if spec['dim'] != 'age_group' else list(spec['filters']), 'age_band': group if spec['dim'] == 'age_group' else None}
            gl_vi, gl_en = stats.group_label(dim_label[0], group, 'vi'), stats.group_label(dim_label[0], group, 'en')
            who_vi = f'người thuộc {DIMENSIONS[spec["dim"]]["vi"]} {gl_vi}' if spec['dim'] != 'age_group' else f'người thuộc nhóm tuổi {group}'
            who_en = f'people in the {DIMENSIONS[spec["dim"]]["en"]} {gl_en}' if spec['dim'] != 'age_group' else f'people aged {group}'
            tgt_vi, tgt_en = stats.target_phrase(spec['target'], 'vi'), stats.target_phrase(spec['target'], 'en')
            return {'metric': 'rate', 'value': pct, 'unit': '%', 'numerator': k, 'denominator': n, 'group': str(group),
                    'label_vi': f'Tỉ lệ người {tgt_vi} trong {who_vi} (nhóm có tỉ lệ {"cao" if spec["which"] == "top" else "thấp"} nhất)',
                    'label_en': f'Share of people {tgt_en} among {who_en} (the {"highest" if spec["which"] == "top" else "lowest"} group)',
                    'sentence_vi': f'Nhóm có tỉ lệ {"cao" if spec["which"] == "top" else "thấp"} nhất là {who_vi}: {fmt_pct(pct, "vi")} ({fmt_int(k, "vi")} trong {fmt_int(n, "vi")} người) {tgt_vi}.',
                    'sentence_en': f'The {"highest" if spec["which"] == "top" else "lowest"} rate is among {who_en}: {fmt_pct(pct)} ({fmt_int(k, "en")} of {fmt_int(n, "en")}) {tgt_en}.',
                    'short_vi': f'nhóm tuổi {group}' if spec['dim'] == 'age_group' else f'{DIMENSIONS[spec["dim"]]["vi"]} {gl_vi}', 'short_en': f'the {group} age group' if spec['dim'] == 'age_group' else f'the {DIMENSIONS[spec["dim"]]["en"]} {gl_en}'}
        df, dvi, den = self.subset(spec)
        n = len(df)
        scope_vi = ' '.join(dvi)
        scope_en = ' '.join(den)
        if spec['kind'] == 'mean_bmi':
            mean = float(df.bmi.mean())
            return {'metric': 'mean_bmi', 'value': mean, 'unit': 'BMI', 'denominator': n,
                    'label_vi': f'BMI trung bình của người {scope_vi}' if scope_vi else 'BMI trung bình của cả mẫu',
                    'label_en': f'Average BMI of people {scope_en}' if scope_en else 'Average BMI of the whole sample',
                    'sentence_vi': f'BMI trung bình của người {scope_vi or "trong cả mẫu"} là {fmt_num(mean, "vi")} (tính trên {fmt_int(n, "vi")} người).',
                    'sentence_en': f'The average BMI of people {scope_en or "in the whole sample"} is {fmt_num(mean, "en")} (over {fmt_int(n, "en")} people).',
                    'short_vi': spec.get('short_vi') or ('nhóm ' + scope_vi if scope_vi else 'cả mẫu'), 'short_en': spec.get('short_en') or ('the group ' + scope_en if scope_en else 'the whole sample')}
        if spec['kind'] == 'count':
            return {'metric': 'count', 'value': float(n), 'unit': 'người', 'denominator': self.stats.n, 'numerator': n,
                    'label_vi': f'Số người {scope_vi}' if scope_vi else 'Số người trong mẫu', 'label_en': f'Number of people {scope_en}' if scope_en else 'Sample size',
                    'sentence_vi': f'Có {fmt_int(n, "vi")} người {scope_vi} trong {fmt_int(self.stats.n, "vi")} người được khảo sát.'.replace('  ', ' '),
                    'sentence_en': f'{fmt_int(n, "en")} of {fmt_int(self.stats.n, "en")} respondents {scope_en}.'.replace('  ', ' '),
                    'short_vi': spec.get('short_vi') or ('nhóm ' + scope_vi if scope_vi else 'cả mẫu'), 'short_en': spec.get('short_en') or ('the group ' + scope_en if scope_en else 'the whole sample')}
        mask = stats.target_mask(df, spec['target'])
        k, pct = int(mask.sum()), float(mask.mean() * 100)
        tgt_vi, tgt_en = stats.target_phrase(spec['target'], 'vi'), stats.target_phrase(spec['target'], 'en')
        pop_vi = f'{fmt_int(n, "vi")} người {scope_vi}' if scope_vi else f'{fmt_int(n, "vi")} người được khảo sát'
        pop_en = f'{fmt_int(n, "en")} respondents {scope_en}' if scope_en else f'{fmt_int(n, "en")} surveyed people'
        return {'metric': 'rate', 'value': pct, 'unit': '%', 'numerator': k, 'denominator': n,
                'label_vi': f'Tỉ lệ người {tgt_vi}' + (f' trong nhóm {scope_vi}' if scope_vi else ' trong cả mẫu'),
                'label_en': f'Share of people {tgt_en}' + (f' among those {scope_en}' if scope_en else ' in the whole sample'),
                'sentence_vi': f'Trong {pop_vi}, {fmt_pct(pct, "vi")} ({fmt_int(k, "vi")} người) {tgt_vi}.',
                'sentence_en': f'Among {pop_en}, {fmt_pct(pct)} ({fmt_int(k, "en")} people) are {tgt_en}.',
                'short_vi': spec.get('short_vi') or ('nhóm ' + scope_vi if scope_vi else 'cả mẫu'), 'short_en': spec.get('short_en') or ('the group ' + scope_en if scope_en else 'the whole sample')}

    def act(self, state):
        t0 = time.perf_counter()
        for spec in state.plan['facts']:
            fact = self.evaluate(spec, state)
            fact.update(id=spec['id'], source=self.stats.source, source_url=self.stats.url, tool='DiabetesStats + pandas', sample=self.stats.n)
            state.facts.append(fact)
        from src.chapter04.stats import fmt_pct, fmt_int, fmt_num
        lines = [f'{f["id"]}: {f["label_vi"]} = {fmt_pct(f["value"], "vi") if f["unit"] == "%" else fmt_num(f["value"], "vi") if f["unit"] == "BMI" else fmt_int(f["value"], "vi")}'
                 + (f' ({fmt_int(f["numerator"], "vi")}/{fmt_int(f["denominator"], "vi")})' if f.get('numerator') is not None and f['unit'] == '%' else '') for f in state.facts]
        state.send('stats', 'calculator' if state.plan['calc'] else 'writer', 'facts', ids=[f['id'] for f in state.facts], values=[f['value'] for f in state.facts])
        state.steps.append({'agent': 'StatsAgent', 'role': self.role, 'role_description': self.description,
                            'summary': f'Agent thống kê đọc dữ liệu CDC BRFSS 2023 ({fmt_int(self.stats.n, "vi")} dòng) và tính {len(state.facts)} số liệu bằng pandas: ' + '; '.join(lines) + '.',
                            'outcome': f'{len(state.facts)} sự kiện', 'seconds': time.perf_counter() - t0, 'evidence': {'facts': state.facts}})

## 4. Agent tính toán

        Lấy hai sự kiện của agent thống kê rồi gọi `safe_calculate` (không dùng `eval`, kiểm tra đơn vị và chia cho 0). Mô hình ngôn ngữ không được tự chia hay trừ.

In [ ]:
# Cell AGENT-05
# ---------------------------------------------------------------------------------------------
# Agent tính toán
# ---------------------------------------------------------------------------------------------
class CalculatorAgent:
    role = 'Agent tính toán'
    description = 'Tính tỉ số, chênh lệch từ các sự kiện bằng safe_calculate, không để mô hình ngôn ngữ tự tính.'

    def act(self, state):
        from src.chapter04.knowledge import safe_calculate
        from src.chapter04.stats import fmt_num
        t0 = time.perf_counter()
        calc = state.plan['calc']
        by_id = {f['id']: f for f in state.facts}
        a, b = by_id[calc['a']], by_id[calc['b']]
        op = calc['op']
        va, vb = a['value'], b['value']
        if op == 'subtract' and va < vb:
            a, b, va, vb = b, a, vb, va
        res = safe_calculate(op, [va, vb], [a['unit'], b['unit']])
        value = res['value']
        unit_vi = 'điểm phần trăm' if (op == 'subtract' and a['unit'] == '%') else ('' if op == 'divide' else 'điểm BMI')
        if op == 'divide':
            sentence_vi = f'Tỉ lệ ở {a["short_vi"]} gấp {fmt_num(value, "vi", 2)} lần ở {b["short_vi"]}.' if a['unit'] == '%' else f'BMI trung bình của {a["short_vi"]} gấp {fmt_num(value, "vi", 2)} lần của {b["short_vi"]}.'
            sentence_en = f'The value for {a["short_en"]} is {fmt_num(value, "en", 2)} times the value for {b["short_en"]}.'
        else:
            sentence_vi = (f'{a["short_vi"][:1].upper() + a["short_vi"][1:]} cao hơn {b["short_vi"]} {fmt_num(value, "vi")} {unit_vi}.').replace('  ', ' ')
            sentence_en = f'{a["short_en"][:1].upper() + a["short_en"][1:]} is higher than {b["short_en"]} by {fmt_num(value, "en")} ' + ('percentage points.' if a['unit'] == '%' else 'BMI points.')
        digits = 2 if op == 'divide' else 1
        derived = {'id': calc['id'], 'metric': op, 'value': round(value, digits + 2), 'unit': 'lần' if op == 'divide' else unit_vi, 'formula': f'{a["id"]} {"/" if op == "divide" else "-"} {b["id"]} = {res["formula"]}',
                   'label_vi': 'Tỉ số' if op == 'divide' else 'Chênh lệch', 'label_en': 'Ratio' if op == 'divide' else 'Difference', 'sentence_vi': sentence_vi, 'sentence_en': sentence_en,
                   'inputs': [a['id'], b['id']], 'digits': digits, 'source': 'safe_calculate (src/chapter04/knowledge.py)', 'tool': 'safe_calculate'}
        state.derived.append(derived)
        state.send('calculator', 'writer', 'derived', id=derived['id'], value=derived['value'], formula=derived['formula'])
        state.steps.append({'agent': 'CalculatorAgent', 'role': self.role, 'role_description': self.description,
                            'summary': f'Agent tính toán lấy {a["id"]} = {fmt_num(va, "vi", 2)} và {b["id"]} = {fmt_num(vb, "vi", 2)} từ agent thống kê rồi gọi safe_calculate: {derived["formula"]}. '
                                       f'Kết quả {fmt_num(value, "vi", derived["digits"])}{(" " + derived["unit"]) if op != "divide" else " lần"}. Mô hình ngôn ngữ không được tự làm phép tính này.',
                            'outcome': f'{derived["label_vi"]} = {fmt_num(value, "vi", derived["digits"])}', 'seconds': time.perf_counter() - t0, 'evidence': {'derived': derived}})

## 5. Agent truy hồi và agent đồ thị

        Agent truy hồi dịch phần giải thích của câu hỏi sang tiếng Anh, truy hồi hybrid (BM25 + vector) và rút hai câu giải thích tốt nhất có mã nguồn. Nếu không đoạn nào chứa từ khóa của truy vấn,
        nó báo kho tài liệu không có thông tin (nhóm sẽ trả lời chỉ bằng số liệu hoặc từ chối). Agent đồ thị chỉ dùng cho tài chính: tra đoạn thuộc báo cáo và công ty nào.

In [ ]:
# Cell AGENT-06
# ---------------------------------------------------------------------------------------------
# Agent truy hồi và agent đồ thị
# ---------------------------------------------------------------------------------------------
def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n+', re.sub(r'\s+', ' ', str(text))) if len(s.strip()) > 25]


class RetrievalAgent:
    role = 'Agent truy hồi'
    description = 'Tìm đoạn tài liệu liên quan (NIDDK, CDC, MedlinePlus) và rút ra câu giải thích ngắn kèm mã nguồn.'

    def __init__(self, engine):
        self.engine = engine

    def act(self, state):
        from src.chapter04.rag import content_tokens, lexical_support
        t0 = time.perf_counter()
        question = state.plan.get('explain_question') or state.question
        if hasattr(self.engine, 'to_english_query'):
            state.query, language = self.engine.to_english_query(question)
        else:
            state.query, language = question, 'en'
        state.language = state.language or language
        found = self.engine.retrieve(state.query, state.domain, k=getattr(self.engine, 'K_EVIDENCE', 3), report=state.report)
        supported = bool(found) and lexical_support(state.query, found)
        state.evidence = found if supported else []
        terms = set(content_tokens(state.query))
        scored = []
        for e in state.evidence:
            for s in split_sentences(e['text']):
                words = set(content_tokens(s))
                scored.append((len(terms & words) / (len(words) ** 0.5 + 1), s, e))
        scored.sort(key=lambda x: -x[0])
        state.explanation = [{'id': e['id'], 'sentence': s, 'score': round(sc, 3), 'url': e.get('source_url'), 'org': e.get('source_org', '')} for sc, s, e in scored[:2] if sc > 0]
        state.send('retrieval', 'writer', 'passages', ids=[e['id'] for e in state.evidence], explanation=len(state.explanation))
        if not state.evidence:
            summary = (f'Agent truy hồi dịch phần giải thích của câu hỏi thành truy vấn "{state.query}" và lấy các đoạn gần nhất, nhưng không đoạn nào chứa từ khóa của truy vấn. '
                       'Nó báo cho nhóm biết kho tài liệu có lẽ không nói về chủ đề này.')
            outcome = 'không tìm thấy tài liệu phù hợp'
        else:
            top = state.evidence[0]
            summary = (f'Agent truy hồi dịch phần giải thích của câu hỏi thành truy vấn "{state.query}" và tìm được {len(state.evidence)} đoạn từ kho tài liệu. '
                       f'Câu giải thích tốt nhất (nguồn {top.get("source_org", "")} {top["id"]}): «{state.explanation[0]["sentence"] if state.explanation else top["text"][:160]}»')
            outcome = f'{len(state.evidence)} đoạn'
        state.steps.append({'agent': 'RetrievalAgent', 'role': self.role, 'role_description': self.description, 'summary': summary, 'outcome': outcome,
                            'seconds': time.perf_counter() - t0, 'evidence': {'query': state.query, 'passages': [{'id': e['id'], 'title': e.get('title', ''), 'url': e.get('source_url'), 'text': str(e['text'])[:300]} for e in state.evidence],
                                                                          'explanation': state.explanation}})


class GraphAgent:
    role = 'Agent đồ thị'
    description = 'Tra đồ thị tri thức để biết mỗi đoạn thuộc báo cáo hay công ty nào.'

    def __init__(self, graphs):
        self.graphs = graphs

    def act(self, state):
        t0 = time.perf_counter()
        graph = self.graphs.get(state.domain)
        labels = []
        if graph is not None:
            for item in state.evidence:
                node = 'passage:' + item['id']
                if node in graph:
                    parents = list(graph.predecessors(node))
                    state.graph_context.append({'id': item['id'], 'parents': parents})
                    for p in parents:
                        data = graph.nodes[p]
                        label = (f"báo cáo {p.split(':')[1].split('/')[0]} năm {data.get('year')}" if data.get('type') == 'Report' else str(p))
                        if label not in labels:
                            labels.append(label)
        state.send('graph', 'writer', 'sources', matched=len(state.graph_context))
        summary = (f'Agent đồ thị tra {len(state.graph_context)} đoạn trong đồ thị tri thức và biết chúng thuộc: {"; ".join(labels)}.' if state.graph_context
                   else 'Agent đồ thị không tìm thấy các đoạn này trong đồ thị nên chuyển tiếp nguyên các đoạn.')
        state.steps.append({'agent': 'GraphAgent', 'role': self.role, 'role_description': self.description, 'summary': summary, 'outcome': f'{len(state.graph_context)} đoạn khớp',
                            'seconds': time.perf_counter() - t0, 'evidence': {'graph': state.graph_context, 'labels': labels}})

## 6. Agent viết

        Bản nháp đầu do Qwen viết tự do từ số của công cụ và các đoạn tài liệu (nó có thể lẫn số trong tài liệu hoặc bịa). Khi bị phản biện, vòng 1 Qwen được nhắc lại cùng danh sách lỗi;
        vòng 2 agent viết dựng lại câu trả lời xác định từ số của công cụ và một câu giải thích trích nguyên văn, nên không thể sai số. Bản đã duyệt được dịch sang tiếng Việt rồi phản biện kiểm lại.

In [ ]:
# Cell AGENT-07
# ---------------------------------------------------------------------------------------------
# Agent viết
# ---------------------------------------------------------------------------------------------
class WriterAgent:
    role = 'Agent viết'
    description = 'Viết câu trả lời từ sự kiện của công cụ và các đoạn tài liệu (bản nháp đầu do mô hình Qwen viết tự do).'

    SYSTEM = ('You are the writer in a team that answers health-data questions for a lay reader. You receive NUMBERS from a statistics tool and PASSAGES from health websites. '
              'Write 3 to 4 plain sentences in English that answer the question. Use the tool numbers exactly as written and say they come from the survey sample. '
              'Use the passages only for explanations; if you mention a number from a passage, say it comes from the documents. Do not invent numbers.')

    def __init__(self, generator):
        self.generator = generator

    # --- dựng đầu vào cho bộ sinh
    def facts_block(self, state):
        lines = [f'{f["id"]}. {f["sentence_en"]}' for f in state.facts] + [f'{d["id"]}. {d["sentence_en"]}' for d in state.derived]
        return '\n'.join(lines)

    def english_question(self, state):
        return state.query if state.plan.get('mode') == 'simple' and state.query else state.question

    def prompt(self, state, feedback=None):
        parts = []
        if state.facts:
            n = state.facts[0].get('sample', 150000)
            parts.append(f'NUMBERS FROM THE STATISTICS TOOL (CDC BRFSS 2023 survey sample of {n:,} adults, unweighted):\n' + self.facts_block(state))
        if state.evidence:
            parts.append('PASSAGES:\n' + '\n---\n'.join(str(e['text'])[:900] for e in state.evidence[:3]))
        q = state.question if state.facts else (state.query or state.question)
        parts.append(f'Question: {q}')
        if feedback:
            parts.append('Your previous answer was reviewed and had these problems:\n' + '\n'.join('- ' + f for f in feedback)
                         + '\nRewrite the answer and fix every problem. Use the tool numbers exactly as written.')
        parts.append('Answer:')
        return '\n\n'.join(parts)

    def llm_draft(self, state, feedback=None):
        if not hasattr(self.generator, 'chat'):
            return None
        text = self.generator.chat(self.SYSTEM, self.prompt(state, feedback), max_new_tokens=200)
        return re.sub(r'\[\d+\]', '', text).strip()

    def rebuild(self, state):
        """Dựng lại bản nháp từ số của công cụ và câu giải thích (xác định, không dùng mô hình)."""
        parts = [f['sentence_en'] for f in state.facts] + [d['sentence_en'] for d in state.derived]
        if state.explanation:
            e = state.explanation[0]
            parts.append(f'According to the documents ({e["org"] or "source"}): {e["sentence"]}')
        if state.facts:
            parts.append(SAMPLE_SIZE_NOTE_EN.format(n=f'{state.facts[0].get("sample", 150000):,}'))
        return ' '.join(parts)

    def rebuild_vi(self, state, translate=True):
        from src.chapter04.stats import fmt_int
        parts = [f['sentence_vi'] for f in state.facts] + [d['sentence_vi'] for d in state.derived]
        if state.explanation:
            e = state.explanation[0]
            sentence = e['sentence']
            if translate and hasattr(self.generator, 'to_vietnamese'):
                sentence = self.generator.to_vietnamese(sentence)[0]
            parts.append(f'Theo tài liệu ({e["org"] or "nguồn"}): {sentence}')
        if state.facts:
            parts.append(SAMPLE_SIZE_NOTE_VI.format(n=fmt_int(state.facts[0].get('sample', 150000), 'vi')))
        return ' '.join(parts)

    def extractive(self, state):
        if not state.evidence:
            return ''
        if state.explanation:
            return ' '.join(f'According to the documents ({e["org"] or "source"}): {e["sentence"]}' for e in state.explanation[:2])
        return 'According to the documents: ' + split_sentences(state.evidence[0]['text'])[0]

    def act(self, state, round_no=0, feedback=None):
        t0 = time.perf_counter()
        from src.chapter04.rag import INSUFFICIENT_EN, INSUFFICIENT_VI
        vi = state.language == 'vi'
        if round_no == 0:
            if state.plan['mode'] == 'simple' and not state.evidence:
                state.status = 'insufficient_evidence'
                state.answer = INSUFFICIENT_VI if vi else INSUFFICIENT_EN
                state.steps.append({'agent': 'WriterAgent', 'role': self.role, 'role_description': self.description, 'seconds': time.perf_counter() - t0,
                                    'summary': 'Agent viết thấy agent truy hồi không tìm được đoạn nào chứa câu trả lời, nên không viết gì và báo lại cho điều phối.', 'outcome': 'từ chối viết', 'evidence': {}})
                state.send('writer', 'coordinator', 'refusal')
                return None
            text = self.llm_draft(state)
            by = 'Qwen viết tự do'
            if not text or self.generator_refused(text):
                if state.facts or state.evidence:
                    text, by = self.rebuild(state), 'dựng từ sự kiện (bộ sinh không viết được)'
        elif round_no == 1:
            text = self.llm_draft(state, feedback)
            by = 'Qwen sửa theo phản biện'
            if not text:
                text, by = self.rebuild(state), 'dựng lại từ số của công cụ'
        else:
            text = self.rebuild(state) if (state.facts or state.explanation) else self.extractive(state)
            by = 'dựng lại từ số của công cụ' if state.facts else 'trích nguyên câu từ tài liệu'
        state.drafts.append({'round': round_no, 'by': by, 'text': text})
        state.send('writer', 'critic', 'draft', round=round_no, by=by, text=text)
        label = 'bản nháp' if round_no == 0 else f'bản sửa lần {round_no}'
        state.steps.append({'agent': 'WriterAgent', 'role': self.role, 'role_description': self.description, 'seconds': time.perf_counter() - t0,
                            'summary': f'Agent viết soạn {label} ({by}): «{text}»', 'outcome': label, 'evidence': {'draft': text, 'round': round_no}})
        return text

    @staticmethod
    def generator_refused(text):
        low = text.strip().lower()
        return low.startswith('insufficient') or 'not enough information' in low

    def finalize(self, state, text):
        """Dịch bản nháp đã được duyệt sang tiếng Việt khi câu hỏi bằng tiếng Việt. Trả về (văn bản, lỗi dịch)."""
        if state.language != 'vi':
            return text, []
        if not hasattr(self.generator, 'to_vietnamese'):
            return text, []
        return self.generator.to_vietnamese(text)

## 7. Agent phản biện

        Không dùng mô hình ngôn ngữ. Các kiểm tra: (a) mỗi con số trong bản nháp phải khớp một sự kiện của công cụ (cho phép làm tròn) hoặc một số trong đoạn tài liệu;
        số chỉ có trong tài liệu mà câu lại nói về mẫu khảo sát, hay không nói là lấy từ tài liệu, bị gắn cờ; (b) bản nháp phải chứa mọi số kết quả chính; (c) phải nhắc rằng đây là số liệu của mẫu;
        (d) mã nguồn trích dẫn phải có thật. Giới hạn: không kiểm ngữ nghĩa (câu "huyết áp cao làm tăng nguy cơ" có đúng ý tài liệu hay không).

In [ ]:
# Cell AGENT-08
# ---------------------------------------------------------------------------------------------
# Agent phản biện: bộ kiểm tra chạy được, không phải mô hình ngôn ngữ
# ---------------------------------------------------------------------------------------------
class CriticAgent:
    role = 'Agent phản biện'
    description = 'Đối chiếu từng con số trong bản nháp với sự kiện của công cụ và với đoạn tài liệu; không dùng mô hình ngôn ngữ.'

    def __init__(self, corpora=None):
        self.allowed = {d: {x['id'] for x in c} for d, c in (corpora or {}).items()}

    def check(self, text, state, language=None):
        """Trả về dict: ok, issues, numbers (từng số kèm nguồn)."""
        from src.chapter04.stats import fmt_pct, fmt_num, fmt_int
        issues, numbers = [], []
        known = fact_numbers(state)
        passage_text = ' '.join(str(e['text']) for e in state.evidence)
        passage_nums = extract_numbers(passage_text)
        has_facts = bool(state.facts)
        for item in extract_numbers(text, ignore=[state.question]):
            hit = next(((v, fid, f) for v, fid, f in known if close(item['value'], item['decimals'], item['scale'], v)), None)
            if hit:
                numbers.append({**{k: item[k] for k in ('text', 'value')}, 'verdict': 'ok', 'source': f'công cụ {hit[1]}', 'sentence': item['sentence']})
                continue
            if item['value'] in (150000.0, 2023.0):
                numbers.append({'text': item['text'], 'value': item['value'], 'verdict': 'ok', 'source': 'cỡ mẫu / năm của nguồn dữ liệu', 'sentence': item['sentence']})
                continue
            doc = next((p for p in passage_nums if close(item['value'], item['decimals'], item['scale'], p['value']) or close(p['value'], p['decimals'], p['scale'], item['value'])), None)
            if doc:
                misused = bool(SAMPLE_MARK.search(item['sentence'])) and has_facts
                unattributed = has_facts and not DOC_MARK.search(item['sentence'])
                if misused or unattributed:
                    why = 'số này nằm trong tài liệu (một quần thể khác) nhưng được viết như số liệu của mẫu khảo sát' if misused else 'số này chỉ có trong tài liệu nhưng câu không nói là lấy từ tài liệu'
                    right = '; '.join(f'{f["id"]}: {f["sentence_vi"]}' for f in state.facts[:2])
                    issues.append({'type': 'number_from_document', 'number': item['text'], 'sentence': item['sentence'],
                                   'message': f'Số {item["text"]}: {why}. Bằng chứng đúng của công cụ: {right}', 'message_en': f'The number {item["text"]} comes from a document, not from the tool. Use the tool numbers.'})
                    numbers.append({'text': item['text'], 'value': item['value'], 'verdict': 'sai nguồn', 'source': 'tài liệu', 'sentence': item['sentence']})
                else:
                    numbers.append({'text': item['text'], 'value': item['value'], 'verdict': 'ok', 'source': 'đoạn tài liệu ' + (state.evidence[0]['id'] if state.evidence else ''), 'sentence': item['sentence']})
                continue
            right = '; '.join(f'{f["id"]}: {f["sentence_vi"]}' for f in state.facts[:3]) if has_facts else 'không có công cụ nào cung cấp số này'
            issues.append({'type': 'unsupported_number', 'number': item['text'], 'sentence': item['sentence'],
                           'message': f'Số {item["text"]} không có trong bằng chứng (không khớp công cụ hay đoạn tài liệu). Bằng chứng đúng: {right}',
                           'message_en': f'The number {item["text"]} is not in the evidence. Correct tool numbers: ' + ' '.join(f['sentence_en'] for f in state.facts[:3])})
            numbers.append({'text': item['text'], 'value': item['value'], 'verdict': 'không có bằng chứng', 'source': '-', 'sentence': item['sentence']})
        # số kết quả chính phải xuất hiện
        written = extract_numbers(text, ignore=[state.question])
        for f in state.facts + state.derived:
            digits = f.get('digits', 1)
            ok = any(close(w['value'], w['decimals'], w['scale'], f['value']) for w in written)
            if not ok:
                shown = fmt_pct(f['value'], 'vi') if f.get('unit') == '%' else fmt_num(f['value'], 'vi', digits)
                issues.append({'type': 'missing_headline', 'number': shown, 'sentence': '',
                               'message': f'Thiếu số kết quả chính {f["id"]} = {shown}. {f["sentence_vi"]}', 'message_en': f'Missing key number: {f["sentence_en"]}'})
        if has_facts and not LIMIT_MARK.search(text):
            issues.append({'type': 'missing_limit', 'number': '', 'sentence': '', 'message': 'Thiếu câu nói rõ đây là số liệu của mẫu khảo sát (không đại diện cho toàn dân).',
                           'message_en': 'Say that the numbers describe the survey sample and not the whole population.'})
        cited = [e['id'] for e in state.evidence]
        valid = all(c in self.allowed.get(state.domain, set()) for c in cited) if self.allowed else True
        if not valid:
            issues.append({'type': 'bad_citation', 'number': '', 'sentence': '', 'message': 'Có mã nguồn trích dẫn không tồn tại trong kho tài liệu.', 'message_en': 'A citation id is not in the corpus.'})
        return {'ok': not issues, 'issues': issues, 'numbers': numbers, 'citations_valid': valid, 'checked_numbers': len(numbers)}

    def act(self, state, text, round_no, language=None):
        t0 = time.perf_counter()
        result = self.check(text, state)
        result['round'] = round_no
        state.reviews.append(result)
        if result['ok']:
            state.send('critic', 'coordinator', 'approval', round=round_no, numbers=result['checked_numbers'])
            summary = (f'Agent phản biện trích {result["checked_numbers"]} con số trong bản nháp và đối chiếu từng số với công cụ và tài liệu: tất cả đều khớp; '
                       f'mã nguồn trích dẫn hợp lệ{"; câu trả lời có nhắc giới hạn của mẫu" if state.facts else ""}. Chấp nhận.')
            outcome = 'chấp nhận'
        else:
            state.send('critic', 'writer', 'revision_request', round=round_no, issues=[i['message'] for i in result['issues']])
            summary = (f'Agent phản biện đối chiếu bản nháp và tìm thấy {len(result["issues"])} lỗi: ' + ' | '.join(i['message'] for i in result['issues'][:4]) + '. Gửi yêu cầu sửa cho agent viết.')
            outcome = f'{len(result["issues"])} lỗi, yêu cầu sửa'
        state.steps.append({'agent': 'CriticAgent', 'role': self.role, 'role_description': self.description, 'summary': summary, 'outcome': outcome,
                            'seconds': time.perf_counter() - t0, 'evidence': {'review': result}})
        return result

## 8. Bộ điều phối và vòng phản biện

        `MultiAgentTeam.ask` chạy kế hoạch: thống kê, tính toán, truy hồi, đồ thị, viết nháp, rồi vòng "phản biện, viết lại" với tối đa 2 vòng sửa. Điều kiện dừng: phản biện chấp nhận, hoặc hết vòng
        (trả lời kèm cảnh báo). Cuối cùng bản tiếng Việt được kiểm lại một lần nữa và bộ điều phối kết luận trạng thái bằng lời.

In [ ]:
# Cell AGENT-09
# ---------------------------------------------------------------------------------------------
# Bộ điều phối: chạy kế hoạch, vòng phản biện, kết luận
# ---------------------------------------------------------------------------------------------
CONCLUSIONS = {
    'verified_numbers': ('Đã kiểm chứng số liệu bằng công cụ', 'Mọi con số trong câu trả lời đã được đối chiếu với kết quả của công cụ thống kê/tính toán (và với tài liệu cho phần giải thích). Phần lời giải thích chưa được kiểm tra về ngữ nghĩa.'),
    'generated_unverified': ('Đã viết, chưa kiểm chứng nội dung', 'Câu hỏi chỉ cần tri thức từ tài liệu: các con số (nếu có) khớp đoạn tài liệu, nhưng hệ thống chưa kiểm tra được lời giải thích có đúng ý nguồn hay không.'),
    'warnings': ('Trả lời kèm cảnh báo', 'Sau hai vòng sửa, agent phản biện vẫn còn nghi ngờ một số điểm; câu trả lời được giữ lại kèm cảnh báo, hãy đối chiếu thủ công.'),
    'insufficient_evidence': ('Không đủ bằng chứng', 'Kho tài liệu và dữ liệu khảo sát không chứa câu trả lời, nên hệ thống từ chối thay vì bịa.'),
    'pending': ('Đang chờ', 'Chưa chạy xong.'),
    'error': ('Lỗi', 'Hệ thống gặp lỗi khi chạy.'),
}


class MultiAgentTeam:
    def __init__(self, engine, graphs, stats):
        self.engine = engine
        self.stats = stats
        self.planner = PlannerAgent(stats)
        self.stats_agent = StatsAgent(stats)
        self.calculator = CalculatorAgent()
        self.retrieval = RetrievalAgent(engine)
        self.graph = GraphAgent(graphs)
        self.writer = WriterAgent(engine.generator)
        self.critic = CriticAgent(engine.corpora)
        self.agents = [self.planner, self.stats_agent, self.calculator, self.retrieval, self.graph, self.writer, self.critic]

    def ask(self, question, domain, report=None, emit=None):
        if domain not in ('diabetes', 'finance'):
            raise ValueError('Unsupported domain')
        if not question.strip():
            raise ValueError('Empty question')
        state = State(question, domain, report)
        state.language = 'vi' if re.search('[à-ỹ]', question.lower()) or re.search(r'\b(bao nhieu|ti le|tieu duong)\b', norm(question)) else 'en'
        seen = [0]

        def flush():
            while emit and seen[0] < len(state.steps):
                emit(state.steps[seen[0]], state)
                seen[0] += 1
        self.planner.act(state)
        flush()
        agents = state.plan['agents']
        if state.plan['scope'] == 'off_topic':
            self.conclude(state, 'insufficient_evidence', refuse=True)
            flush()
            return state
        if 'StatsAgent' in agents:
            self.stats_agent.act(state); flush()
        if 'CalculatorAgent' in agents:
            self.calculator.act(state); flush()
        if 'RetrievalAgent' in agents:
            self.retrieval.act(state); flush()
        if 'GraphAgent' in agents:
            self.graph.act(state); flush()
        if state.plan['mode'] == 'simple' and not state.evidence:
            self.writer.act(state); flush()
            self.conclude(state, 'insufficient_evidence', refuse=True)
            flush()
            return state
        if state.plan['mode'] == 'data' and not state.facts:
            self.conclude(state, 'insufficient_evidence', refuse=True)
            flush()
            return state
        text = self.writer.act(state, 0); flush()
        review = self.critic.act(state, text, 0); flush()
        revisions = 0
        while not review['ok'] and revisions < MAX_REVISIONS:
            revisions += 1
            feedback = [i['message_en'] for i in review['issues']]
            state.dialogue.append({'round': revisions, 'issues': review['issues']})
            text = self.writer.act(state, revisions, feedback); flush()
            review = self.critic.act(state, text, revisions); flush()
        state.revisions = revisions
        # dịch sang tiếng Việt rồi soát lại bản dịch bằng chính bộ phản biện
        t0 = time.perf_counter()
        final, problems = self.writer.finalize(state, text)
        state.english_answer = text if state.language == 'vi' else ''
        note = ''
        if state.language == 'vi':
            check = self.critic.check(final, state)
            if not check['ok'] and state.facts:
                final = self.writer.rebuild_vi(state)
                note = 'Bản dịch làm lệch số liệu nên agent viết dựng lại bản tiếng Việt trực tiếp từ số của công cụ.'
                check = self.critic.check(final, state)
            elif not check['ok'] and not state.facts and state.evidence:
                note = 'Bản dịch có số không khớp tài liệu; giữ nguyên văn tiếng Anh đã được duyệt.'
                final = text
                check = {'ok': True}
            review = check if not check['ok'] else review
            state.steps.append({'agent': 'WriterAgent', 'role': self.writer.role, 'role_description': self.writer.description, 'seconds': time.perf_counter() - t0,
                                'summary': 'Agent viết dịch bản đã được duyệt sang tiếng Việt, rồi agent phản biện kiểm lại các con số của bản dịch. ' + (note or 'Các con số vẫn khớp.'),
                                'outcome': 'bản tiếng Việt', 'evidence': {'answer': final}})
            flush()
        state.answer = final
        passed = review['ok'] if isinstance(review, dict) else True
        status = ('verified_numbers' if state.facts else 'generated_unverified') if passed else 'warnings'
        self.conclude(state, status)
        flush()
        return state

    def conclude(self, state, status, refuse=False):
        from src.chapter04.rag import INSUFFICIENT_EN, INSUFFICIENT_VI
        state.status = status
        if refuse:
            if state.plan.get('scope') == 'off_topic':
                state.answer = ('Câu hỏi này nằm ngoài phạm vi tiểu đường và tài chính của hệ thống, nên mình không trả lời.' if state.language == 'vi'
                                else 'This question is outside the diabetes and finance scope of the system, so I will not answer it.')
            elif not state.answer:
                state.answer = INSUFFICIENT_VI if state.language == 'vi' else INSUFFICIENT_EN
        revisions = getattr(state, 'revisions', 0)
        label, explain = CONCLUSIONS[status]
        if revisions and status in ('verified_numbers', 'generated_unverified'):
            explain += f' Bản nháp đầu bị phản biện yêu cầu sửa; câu trả lời cuối là bản sau {revisions} vòng sửa.'
        state.send('coordinator', 'user', 'verdict', status=status, label=label, revisions=revisions)
        state.steps.append({'agent': 'Coordinator', 'role': 'Bộ điều phối', 'role_description': 'Tổng hợp trạng thái cuối bằng lời.', 'summary': f'{label}. {explain}',
                            'outcome': label, 'seconds': 0.0, 'evidence': {'status': status, 'revisions': revisions}})
        state.revisions = revisions


def build_dialogue(state):
    """Cuộc trao đổi viết - phản biện, theo thứ tự thời gian, để giao diện hiển thị như hội thoại."""
    out = []
    for d in state.drafts:
        out.append({'speaker': 'Agent viết', 'kind': 'draft' if d['round'] == 0 else 'revision', 'round': d['round'], 'by': d['by'],
                    'text': d['text']})
        review = next((r for r in state.reviews if r['round'] == d['round']), None)
        if review:
            if review['ok']:
                out.append({'speaker': 'Agent phản biện', 'kind': 'approval', 'round': d['round'], 'ok': True,
                            'text': f'Đã đối chiếu {review["checked_numbers"]} con số với công cụ và tài liệu: tất cả khớp. Chấp nhận.',
                            'issues': []})
            else:
                out.append({'speaker': 'Agent phản biện', 'kind': 'objection', 'round': d['round'], 'ok': False,
                            'text': f'Phát hiện {len(review["issues"])} lỗi trong bản này.', 'issues': [i['message'] for i in review['issues']]})
    return out


def compare_baseline(text, state):
    """So câu trả lời của RAG đơn với các số do công cụ tính: có đủ số đúng không, có số lạ không."""
    heads = state.facts + state.derived
    if not heads:
        return {'comparable': False}
    written = extract_numbers(text, ignore=[state.question])
    matched, missing = 0, []
    for f in heads:
        if any(close(w['value'], w['decimals'], w['scale'], f['value']) for w in written):
            matched += 1
        else:
            missing.append(f['sentence_vi'])
    foreign = [w['text'] for w in written if not any(close(w['value'], w['decimals'], w['scale'], v) for v, _, _ in fact_numbers(state)) and w['value'] not in (150000.0, 2023.0)]
    return {'comparable': True, 'matched': matched, 'total': len(heads), 'has_all_numbers': matched == len(heads), 'missing': missing, 'foreign_numbers': foreign}

## 9. Khởi tạo hệ thống

        Nạp kho tài liệu, hai bộ truy hồi, mô hình Qwen, đồ thị tri thức và dữ liệu BRFSS. Mô hình chiếm 6 GB RAM (hoặc khoảng 2,5 GB ở chế độ int8), nên hãy dừng máy chủ 8044 trước khi chạy notebook này.

In [ ]:
# Cell AGENT-10
from deployment.chapter04.app import rag, agent
from src.chapter04.knowledge import load_graph
from scripts.tieuluan.complete_agents import MultiAgentTeam, State, build_dialogue, compare_baseline, CriticAgent, CONCLUSIONS
graphs = {'finance':load_graph(ROOT/'knowledge_graph/chapter04/finance_graph.json.gz'),
          'diabetes':load_graph(ROOT/'knowledge_graph/chapter04/diabetes_documents_graph.json.gz')}
engine = rag(); engine.generator.load()
team = MultiAgentTeam(engine, graphs, agent().stats)
print('Mô hình sinh:', engine.model_name, '| kho tài liệu tiểu đường:', len(engine.corpora['diabetes']), 'đoạn')
print('Các agent:', [type(a).__name__ for a in team.agents])

## 10. Một câu phức hợp chạy đủ các agent

        Câu hỏi cần ba loại việc: đếm trên dữ liệu (tỉ lệ ở nhóm huyết áp cao và ở cả mẫu), phép chia chính xác, và lời giải thích từ tài liệu. Mỗi agent kể lại việc của mình; bên dưới là cuộc trao đổi viết và phản biện.

In [ ]:
# Cell AGENT-11
question = 'Tỉ lệ tiểu đường ở người huyết áp cao gấp mấy lần cả mẫu, và vì sao huyết áp cao làm tăng nguy cơ?'
t0 = time.perf_counter(); state = team.ask(question, 'diabetes'); team_seconds = time.perf_counter()-t0
print('KẾ HOẠCH:', state.plan['mode_label'], '|', state.plan['summary'], '\n')
for step in state.steps:
    print(f"[{step['role']}] ({step['seconds']:.1f} giây) {step['summary'][:700]}")
    print('    =>', step['outcome'], '\n')
print('CUỘC TRAO ĐỔI VIẾT - PHẢN BIỆN:')
for m in build_dialogue(state):
    print(' ', m['speaker'], f"[vòng {m['round']}]", m['text'])
    for issue in m.get('issues', []): print('      -', issue)
print('\nCÂU TRẢ LỜI CUỐI:', state.answer)
print('TRẠNG THÁI:', CONCLUSIONS[state.status][0], '| số vòng sửa:', state.revisions, '| tổng', round(team_seconds), 'giây')

## 11. So với một RAG đơn trên cùng câu hỏi

        RAG đơn (như Chương 5) chỉ truy hồi rồi viết. Nó không có số liệu từ dữ liệu khảo sát, nên hoặc từ chối hoặc trích một con số của tài liệu (về cả nước Mỹ) mà không liên quan.
        Hàm `compare_baseline` kiểm xem câu trả lời có chứa các số đúng do công cụ tính hay không.

In [ ]:
# Cell AGENT-12
t0 = time.perf_counter(); single = engine.answer(question, 'diabetes'); single_seconds = time.perf_counter()-t0
verdict = compare_baseline(single['answer'], state)
print('MỘT RAG ĐƠN (%.0f giây, 2 bước):' % single_seconds, single['answer'])
print('Số đúng có trong câu trả lời:', verdict['matched'], '/', verdict['total'], '| số không đến từ dữ liệu:', verdict['foreign_numbers'])
print('NHÓM AGENT (%.0f giây, %d bước):' % (team_seconds, len(state.steps)), state.answer)

## 12. Kiểm thử agent phản biện bằng bản nháp sai cho trước

        Để thấy cách phản biện hoạt động, ô này đưa cho nó một bản nháp sai do **người viết notebook tạo ra** (không phải đầu ra của mô hình): lấy "37,3 triệu" trong tài liệu NIDDK làm số của mẫu
        và nêu tỉ số sai. Phản biện phải gắn cờ từng số, kèm bằng chứng đúng của công cụ. Đây là kiểm thử đơn vị của bộ kiểm tra; các lỗi do Qwen viết thật nằm ở phần đánh giá bên dưới.

In [ ]:
# Cell AGENT-13
probe = State(question, 'diabetes'); probe.language = 'vi'
from scripts.tieuluan.complete_agents import PlannerAgent, StatsAgent, CalculatorAgent
PlannerAgent(agent().stats).act(probe); StatsAgent(agent().stats).act(probe); CalculatorAgent().act(probe)
probe.evidence = state.evidence
wrong = ('In the survey sample, 37.3 million people with high blood pressure have diabetes, which is 2.9 times the overall rate.')
result = CriticAgent(engine.corpora).check(wrong, probe)
for item in result['numbers']: print('SỐ', item['text'], '->', item['verdict'])
for issue in result['issues']: print(' -', issue['type'], ':', issue['message'])
assert not result['ok']

## 13. Câu đơn giản và câu ngoài phạm vi

        Câu "Insulin do cơ quan nào tạo ra?" chỉ cần đọc tài liệu: kế hoạch chỉ gọi truy hồi và viết, nhóm agent không có thêm bằng chứng nào để kiểm chứng, và chậm hơn hoặc bằng một RAG đơn.
        Câu "Giá vàng hôm nay?" bị agent điều phối từ chối ngay.

In [ ]:
# Cell AGENT-14
rows = []
for q in ['Insulin do cơ quan nào trong cơ thể tạo ra?', 'Giá vàng hôm nay là bao nhiêu?']:
    t0 = time.perf_counter(); st = team.ask(q, 'diabetes'); ts = time.perf_counter()-t0
    t0 = time.perf_counter(); rg = engine.answer(q, 'diabetes'); rs = time.perf_counter()-t0
    rows.append({'câu hỏi':q, 'kế hoạch':st.plan['mode_label'], 'nhóm agent':st.answer[:140], 'trạng thái nhóm':st.status, 'giây nhóm':round(ts),
                 'một RAG đơn':rg['answer'][:140], 'trạng thái RAG':rg['status'], 'giây RAG':round(rs)})
display(pd.DataFrame(rows).T)

## 14. Đánh giá định lượng

        `scripts/chapter05/eval_multiagent.py` chạy 21 câu phức hợp, 5 câu đơn giản và 4 câu ngoài phạm vi qua cả hai hệ (qua API của máy chủ 8044). Đáp án của câu phức hợp được tính độc lập bằng pandas
        từ CSV (không dùng mã của agent thống kê). Một số được coi là đúng khi sai không quá 0,5 điểm phần trăm (tỉ lệ), 1% tương đối (tỉ số, số đếm) hoặc 0,1 (BMI). Bản đầy đủ chạy rất lâu (hàng chục phút đến hàng giờ trên CPU),
        nên notebook đọc kết quả từ `results/chapter05/multiagent_eval.json` thay vì chạy lại; chạy lại bằng `python scripts/chapter05/eval_multiagent.py --resume`.

In [ ]:
# Cell AGENT-15
from scripts.chapter05.eval_multiagent import print_table, summarize
data = json.loads((ROOT/'results/chapter05/multiagent_eval.json').read_text(encoding='utf8'))
print('Số câu đã chấm:', data['questions'])
print_table(data['summary'], data['questions'])
table = pd.DataFrame([{'id':r['id'], 'loại':r['kind'], 'RAG đơn đúng':r['rag']['correct'], 'nhóm agent đúng':r['team']['correct'], 'nháp đầu đúng':r['team'].get('draft_headline_correct'),
                       'vòng sửa':r['team']['revisions'], 'giây RAG':round(r['rag']['seconds'] or 0), 'giây nhóm':round(r['team']['seconds'] or 0)} for r in data['results']])
display(table)

## 15. Phân tích: phản biện đã bắt lỗi thật nào?

        Dưới đây là các câu mà bản nháp đầu của Qwen bị phản biện bác (nếu có), kèm lỗi cụ thể và bản cuối. Nếu không có câu nào bị bác thì ô in ra như vậy: khi bản nháp đúng, phản biện chấp nhận ngay.

In [ ]:
# Cell AGENT-16
caught = [r for r in data['results'] if r['team'].get('draft_review_ok') is False]
print('Số câu có bản nháp đầu bị bác:', len(caught), '/', len(data['results']))
for r in caught[:4]:
    print('\n', r['id'], r['question'])
    print('  Bản nháp đầu:', r['team'].get('draft_text'))
    for issue in r['team'].get('draft_issues', []): print('  Phản biện:', issue)
    print('  Số vòng sửa:', r['team']['revisions'], '| trạng thái cuối:', r['team']['status'])
    print('  Câu trả lời cuối:', r['team']['answer'])

## Câu hỏi trình bày

        1. Vì sao câu "Insulin do cơ quan nào tạo ra?" không cần nhiều agent, còn câu về huyết áp cao thì cần? Chỉ ra ba loại năng lực mà RAG đơn không có.
        2. Agent phản biện khác một mô hình tự kiểm tra ở điểm nào? Nó bắt được loại lỗi nào, và loại lỗi nào nó bỏ sót (gợi ý: một câu giải thích sai ý tài liệu nhưng không chứa số)?
        3. Điều kiện dừng của vòng phản biện là gì và vì sao cần giới hạn 2 vòng?
        4. Trong bảng đánh giá, nhóm agent hơn RAG đơn ở loại câu nào, hòa hoặc thua ở loại câu nào, và vì sao?